# Clean an image batch with SciPy and fastscipy

Remove salt-and-pepper noise from a reproducible sequence of **1024 × 1024 grayscale frames**
with a **3 × 3 median filter**. The notebook sizes the batch to target about **five seconds
of ordinary SciPy filtering**, then measures both implementations on every identical frame.

**Run → Run All Cells** to try it. No data downloads. Frames are generated and checked one at
a time, so a larger batch increases runtime without keeping more images in memory.

## Set up

Use Python 3.12–3.15. In a fresh notebook environment, install the published wheel and plotting
dependency, then restart the kernel if packages were already imported:

```python
%pip install --only-binary=:all: fastscipy==0.1.0a4 matplotlib
```

`import fastscipy` loads the package. **`fastscipy.patch()` enables acceleration** behind the
existing SciPy API. This example uses one native thread; unsupported calls retain SciPy's
original implementation. We explicitly check that the median-filter fast path is used.

In [ ]:
import os

# Apply before numerical imports. Restart the kernel after changing these limits.
for name in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[name] = "1"
os.environ["FASTSCIPY_NUM_THREADS"] = "1"

In [ ]:
from math import ceil
from statistics import median
from time import perf_counter
import platform

import numpy as np
import scipy
from scipy import ndimage
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
import fastscipy

patch_status = fastscipy.patch(threads=1)
assert patch_status.get("ndimage") == "on", patch_status

print(f"{platform.system()} {platform.machine()} · Python {platform.python_version()}")
print(f"NumPy {np.__version__} · SciPy {scipy.__version__} · fastscipy {fastscipy.__version__}")
print(f"fastscipy: {fastscipy.get_num_threads()} native thread")

## Choose a workload

The five-second target is approximate: calibration and machine load affect the final time.
`MAX_FRAMES` caps the amount of work on unusually fast machines. Inputs, outputs, and noise
buffers occupy only a few frames' worth of memory; plotting and imported libraries use more.
Reduce `IMAGE_SIZE` if needed, or set `TARGET_SCIPY_SECONDS = 1.0` for a quicker comparison.

In [ ]:
IMAGE_SIZE = 1024
TARGET_SCIPY_SECONDS = 5.0
MAX_FRAMES = 512
SEED = 42

assert 64 <= IMAGE_SIZE <= 2048, "Keep the demo's memory use modest"
assert 0 < TARGET_SCIPY_SECONDS <= 10
assert 1 <= MAX_FRAMES <= 512

# A smooth background with a round feature and a sharp-edged rectangular feature.
x = np.linspace(0, 1, IMAGE_SIZE, dtype=np.float32)[None, :]
y = np.linspace(0, 1, IMAGE_SIZE, dtype=np.float32)[:, None]
base = 35 + 65 * x + 35 * y
base += 90 * (((x - 0.35) ** 2 + (y - 0.45) ** 2) < 0.13**2)
base += 55 * ((x > 0.58) & (x < 0.85) & (y > 0.25) & (y < 0.75))
base = np.clip(base, 0, 255).astype(np.uint8)
del x, y


def make_frame(index):
    """A deterministic moving frame with 8% salt-and-pepper noise."""
    frame = np.roll(base, shift=(index % 17, 7 * index % IMAGE_SIZE), axis=(0, 1))
    rng = np.random.default_rng(SEED + index)
    noise = rng.random(frame.shape, dtype=np.float32)
    frame[noise < 0.04] = 0
    frame[noise > 0.96] = 255
    return frame


preview = make_frame(0)
print(f"One frame: {IMAGE_SIZE:,} × {IMAGE_SIZE:,}, uint8, {preview.nbytes / 2**20:.1f} MiB")

## Ordinary SciPy

The context manager temporarily disables fastscipy's patches. The call inside it is the
same call an ordinary SciPy application would make.

In [ ]:
with fastscipy.disabled():
    scipy_preview = ndimage.median_filter(preview, size=3, mode="reflect")

## With fastscipy

Outside the disabled context, the same SciPy call uses the accelerated implementation for
these inputs. The counter assertion prevents an accidental SciPy-versus-SciPy comparison.

In [ ]:
fast_before = fastscipy.stats().get(("rank_filter", "fast"), 0)
fast_preview = ndimage.median_filter(preview, size=3, mode="reflect")
assert fastscipy.stats().get(("rank_filter", "fast"), 0) == fast_before + 1, (
    "This input did not use the fast path; inspect fastscipy.status() and fastscipy.stats()"
)
assert scipy_preview.dtype == fast_preview.dtype
assert scipy_preview.shape == fast_preview.shape
assert scipy_preview.tobytes() == fast_preview.tobytes()
print("Acceleration is active; preview outputs are byte-identical.")

## Size the batch

Both implementations are already warmed on the same shape. Three additional plain-SciPy
calls estimate the frame count. These calibration calls are excluded from the result table.

In [ ]:
calibration = []
with fastscipy.disabled():
    for _ in range(3):
        start = perf_counter()
        output = ndimage.median_filter(preview, size=3, mode="reflect")
        calibration.append(perf_counter() - start)
        del output

seconds_per_frame = median(calibration)
frame_count = min(MAX_FRAMES, max(1, ceil(TARGET_SCIPY_SECONDS / seconds_per_frame)))
print(f"Batch: {frame_count} frames; estimated SciPy filtering time {frame_count * seconds_per_frame:.1f} s")

## Time the complete batch

Each frame is created once, then passed to both implementations. Their order alternates
between frames. Timers include **filtering and output allocation**; generation, validation,
progress updates, and rendering are excluded. The table sums filtering time across the batch,
so the cell's total wall time is longer. All frames are checked for byte identity.

This measures one image-cleanup workload, not an average across SciPy. There is no artificial
delay and no minimum-speedup requirement.

In [ ]:
assert fastscipy.status().get("ndimage") == "on", "Run the setup cell first"
counts_before = fastscipy.stats()
scipy_seconds = 0.0
fast_seconds = 0.0
progress = display(Markdown(f"Processing **{frame_count} frames**…"), display_id=True)

for index in range(frame_count):
    frame = make_frame(index)
    order = ("scipy", "fast") if index % 2 == 0 else ("fast", "scipy")
    for implementation in order:
        if implementation == "scipy":
            with fastscipy.disabled():
                start = perf_counter()
                scipy_output = ndimage.median_filter(frame, size=3, mode="reflect")
                scipy_seconds += perf_counter() - start
        else:
            start = perf_counter()
            fast_output = ndimage.median_filter(frame, size=3, mode="reflect")
            fast_seconds += perf_counter() - start

    assert scipy_output.shape == fast_output.shape == frame.shape
    assert scipy_output.dtype == fast_output.dtype == frame.dtype
    assert scipy_output.tobytes() == fast_output.tobytes(), f"Outputs differ on frame {index}"
    del frame, scipy_output, fast_output

    if (index + 1) % max(1, frame_count // 10) == 0:
        progress.update(Markdown(f"Checked **{index + 1} / {frame_count} frames**…"))

counts_after = fastscipy.stats()
fast_calls = counts_after.get(("rank_filter", "fast"), 0) - counts_before.get(("rank_filter", "fast"), 0)
plain_calls = counts_after.get(("rank_filter", "scipy"), 0) - counts_before.get(("rank_filter", "scipy"), 0)
errors = counts_after.get(("rank_filter", "error"), 0) - counts_before.get(("rank_filter", "error"), 0)
assert fast_calls == plain_calls == frame_count, "Unexpected acceleration or fallback count"
assert errors == 0, "The fast path raised an error"

speedup = scipy_seconds / fast_seconds
progress.update(
    Markdown(
        f"### {frame_count} frames · {IMAGE_SIZE:,} × {IMAGE_SIZE:,} pixels\n\n"
        "| Implementation | Total filtering time |\n"
        "|:--|--:|\n"
        f"| Ordinary SciPy | {scipy_seconds:.3f} s |\n"
        f"| fastscipy, 1 thread | {fast_seconds:.3f} s |\n\n"
        f"**{speedup:.1f}× faster** on this run. All **{frame_count} frames are byte-identical**; "
        "every accelerated call used the fast path."
    )
)

## See what changed

Filtering removes isolated bright and dark pixels while retaining the large image features.
Both implementations produce the same result. The crops below use the same coordinates and
intensity range.

In [ ]:
start = IMAGE_SIZE // 3
stop = start + min(192, IMAGE_SIZE // 2)
crop = np.s_[start:stop, start:stop]
# Downsample full-frame previews only; filtering above always uses full resolution.
step = max(1, IMAGE_SIZE // 512)
fig, axes = plt.subplots(2, 2, figsize=(10, 7), constrained_layout=True)
views = (
    (preview[::step, ::step], "Noisy frame"),
    (fast_preview[::step, ::step], "Median-filtered frame"),
    (preview[crop], "Before: detail"),
    (fast_preview[crop], "After: detail"),
)
for ax, (image, title) in zip(axes.flat, views):
    ax.imshow(image, cmap="gray", vmin=0, vmax=255, interpolation="nearest")
    ax.set_title(title)
    ax.axis("off")
plt.show()

## Try your own data

Replace `make_frame` with your image source, keeping each frame as a 2-D `uint8` NumPy array.
Re-run the preview and fast-path check before timing a different workload. Different dtypes,
filter sizes, array layouts, hardware, or thread budgets can change the speedup or select a
SciPy fallback.

[Command-line companion](median_filter.py) · [More examples](README.md) ·
[fastscipy on GitHub](https://github.com/michael-L-i/fastscipy)